In [1]:
import pandas as pd
import numpy as np
from pandas import Series, DataFrame

#### Step 1:

1. Load the temperature data from New York City (from the `nyc-temps.txt` file) into a Series.  
   The measurements are in degrees Celsius.

In [2]:
s = pd.read_csv('../../../data/nyc-temps.txt', header = None, names = ['temp']).squeeze()

#### Step 2:

2. Create a DataFrame with two columns:
   - `temp`, containing the temperatures  
   - `hour`, representing the hours at which the measurements were taken  

   The `hour` values should be:  
   `0, 3, 6, 9, 12, 15, 18, 21`  
   repeated for all 728 data points.

In [3]:
hours = np.arange(0, 24, 3)

df = pd.DataFrame({
    'temp' : s,
    'hour' : np.tile(hours, len(s) // len(hours) +1)[:len(s)]
})

df

,temp,hour
0,-1,0
1,-1,3
2,-1,6
3,-1,9
4,-1,12
...,...,...
724,2,12
725,2,15
726,2,18
727,2,21


#### Step 3:

3. Calculate the __mean__ and __median__ values.  
   These are the real values, which we hope to replicate via interpolation.

In [4]:
temp_real_mean = df['temp'].mean()
temp_real_median = df['temp'].median()

print(
    f'Real mean temperature is {temp_real_mean} ˚C'
    f'\nReal median temperature is {temp_real_median} ˚C'

)

Real mean temperature is -1.0507544581618655 ˚C
Real median temperature is 0.0 ˚C


#### Step 4:

4. Set all values from 3:00 and 6:00 a.m. to `NaN`.

In [5]:
nan_df = df.copy()

In [6]:
nan_df.dtypes

temp    int64
hour    int64
dtype: object

In [7]:
nan_df.loc[(nan_df['hour'] == 3) | (nan_df['hour'] == 6), 'temp'] = np.nan

nan_df

,temp,hour
0,-1.0,0
1,NaN,3
2,NaN,6
3,-1.0,9
4,-1.0,12
...,...,...
724,2.0,12
725,2.0,15
726,2.0,18
727,2.0,21


#### Step 1:

5. Interpolate the values using the `interpolate()` method.

In [8]:
nan_df['temp'] = nan_df['temp'].interpolate()

#### Step 1:

6. Calculate the mean and median of the interpolated DataFrame.  
   - Are they similar to the real values?  
   - Why or why not?

In [9]:
temp_real_mean = df['temp'].mean()
temp_real_median = df['temp'].median()

temp_nan_mean = nan_df['temp'].mean()
temp_nan_median = nan_df['temp'].median()

print(
    f'Real mean temperature is {temp_real_mean} ˚C <-> Interpolated mean temperature is {temp_nan_mean} ˚C'
    f'\nReal median temperature is {temp_real_median} ˚C <-> Interpolated median temperature is {temp_nan_median} ˚C'
)

Real mean temperature is -1.0507544581618655 ˚C <-> Interpolated mean temperature is -1.0507544581618655 ˚C
Real median temperature is 0.0 ˚C <-> Interpolated median temperature is 0.0 ˚C


#### *There is no difference between the mean and median of the original values and the interpolated values, which is understandable since sampling was done only every 3 hours, and temperature changes over this time scale can be interpolated quite well with this amount of missing data.*



### Beyond the exercise_1

##### • How does the behavior of `interpolate` change if you use `method='nearest'`?

In [10]:
nan_df = df.copy()

nan_df.loc[(nan_df['hour'] == 3) | (nan_df['hour'] == 6), 'temp'] = np.nan

nan_df['temp'] = nan_df['temp'].interpolate(method = 'nearest')

temp_real_mean = df['temp'].mean()
temp_real_median = df['temp'].median()

temp_nan_mean = nan_df['temp'].mean()
temp_nan_median = nan_df['temp'].median()

print(
    f'Real mean temperature is {temp_real_mean} ˚C <-> Interpolated mean temperature is {temp_nan_mean} ˚C'
    f'\nReal median temperature is {temp_real_median} ˚C <-> Interpolated median temperature is {temp_nan_median} ˚C'
)

Real mean temperature is -1.0507544581618655 ˚C <-> Interpolated mean temperature is -1.0507544581618655 ˚C
Real median temperature is 0.0 ˚C <-> Interpolated median temperature is 0.0 ˚C


#### *There is no change in the median and mean values; I assume this is because only a small number of nearby data points are missing.*


### Beyond the exercise_2

##### • Let’s assume the equipment works fine around the clock but fails to record readings at –1 degrees and below. Are the interpolated values similar to the real (missing) values they replace? Why or why not?

In [11]:
# LINEAR interpolation test

nan_df = df.copy()

nan_df.loc[nan_df['temp'] <= -1, 'temp'] = np.nan

nan_df['temp'] = nan_df['temp'].interpolate()

temp_real_mean = df['temp'].mean()
temp_real_median = df['temp'].median()

temp_nan_mean = nan_df['temp'].mean()
temp_nan_median = nan_df['temp'].median()

print(
    f'Real mean temperature is {temp_real_mean} ˚C <-> Interpolated mean temperature is {temp_nan_mean} ˚C'
    f'\nReal median temperature is {temp_real_median} ˚C <-> Interpolated median temperature is {temp_nan_median} ˚C'
)

Real mean temperature is -1.0507544581618655 ˚C <-> Interpolated mean temperature is 2.0221914008321775 ˚C
Real median temperature is 0.0 ˚C <-> Interpolated median temperature is 1.0 ˚C


In [12]:
# NEAREST interpolation test


nan_df = df.copy()

nan_df.loc[nan_df['temp'] <= -1, 'temp'] = np.nan

nan_df['temp'] = nan_df['temp'].interpolate(method = 'nearest')

temp_real_mean = df['temp'].mean()
temp_real_median = df['temp'].median()

temp_nan_mean = nan_df['temp'].mean()
temp_nan_median = nan_df['temp'].median()

print(
    f'Real mean temperature is {temp_real_mean} ˚C <-> Interpolated mean temperature is {temp_nan_mean} ˚C'
    f'\nReal median temperature is {temp_real_median} ˚C <-> Interpolated median temperature is {temp_nan_median} ˚C'
)

Real mean temperature is -1.0507544581618655 ˚C <-> Interpolated mean temperature is 2.0221914008321775 ˚C
Real median temperature is 0.0 ˚C <-> Interpolated median temperature is 1.0 ˚C


#### *In this case, interpolation does not work well because many consecutive temperature values are missing, and the effect of distortion becomes more noticeable.*

### Beyond the exercise_3

##### • A cheap solution to interpolation is to replace NaN values with the column’s mean. Do this (with the missing values from –1 and below), and compare the new mean and median. Again, why are (or aren’t) these values similar to the original ones?

In [13]:
# NEAREST interpolation test


nan_df = df.copy()

nan_df.loc[nan_df['temp'] <= -1, 'temp'] = np.nan

nan_df.loc[nan_df['temp'] == np.nan] = nan_df['temp'].mean()

temp_real_mean = df['temp'].mean()
temp_real_median = df['temp'].median()

temp_nan_mean = nan_df['temp'].mean()
temp_nan_median = nan_df['temp'].median()

print(
    f'Real mean temperature is {temp_real_mean} ˚C <-> Interpolated mean temperature is {temp_nan_mean} ˚C'
    f'\nReal median temperature is {temp_real_median} ˚C <-> Interpolated median temperature is {temp_nan_median} ˚C'
)

Real mean temperature is -1.0507544581618655 ˚C <-> Interpolated mean temperature is 2.763925729442971 ˚C
Real median temperature is 0.0 ˚C <-> Interpolated median temperature is 2.0 ˚C


#### *This gives an even worse result.*